# HW 2 — 게임의 기댓값

**확률통계 · Topic 3** | 배점 25점 | 개인 과제

---

## ✍️ 제출자 정보 — 먼저 채우세요

| | |
|---|---|
| **학번** | (여기에 작성) |
| **고른 게임** | (여기에 작성 — 예: "모바일 게임 10연차 가챠") |

> 파일명을 **`HW_학번_T03.ipynb`** 로 바꿔서 제출한다. (예: `HW_202512345_T03.ipynb`)
> ⚠️ **제출 방법과 기한은 PLATO · Google Classroom 공지**를 확인한다.

---

## 무엇을 하는 과제인가

**무작위성이 있는 게임(또는 뽑기) 하나를 직접 골라** 확률 모형으로 기술하고,
기댓값과 분산을 손으로 계산한 뒤 시뮬레이션으로 검증한다.

| 주제 예시 | 확률변수 $X$ 의 예 |
|---|---|
| 모바일 게임 가챠 | 10연차에서 얻는 ★5 개수 |
| 아이템 강화 | 목표 단계까지 필요한 시도 횟수 |
| 로또 · 복권 | 1장당 순수익 |
| 자판기 랜덤 상품 | 원하는 상품을 얻기까지의 비용 |

> ⚠️ 실제 게임의 공시 확률을 쓸 경우 **출처를 명시**할 것. 없으면 가정을 직접 세우고 명시한다.

### 할 일

| 문제 | 내용 | 배점 |
|:-:|---|:-:|
| 1 | 모형 세우기 — 규칙 서술 · $X$ 정의 · PMF 표 | 5 |
| 2 | 손으로 계산 — $\mathbb{E}[X]$ · $\mathbb{E}[X^2]$ · $\mathrm{Var}[X]$ · $\sigma_X$ | 8 |
| 3 | 시뮬레이션 검증 — 10만 회 · 비교표 · 수렴 그래프 | 7 |
| 4 | 해석 (a)(b)(c) 서술 | 5 |

⚠️ **제출 전 `런타임 → 모두 실행`** 으로 출력을 남길 것. 출력이 없으면 −2점.

## Part 0. 준비

이 셀을 먼저 실행한다. **시드는 바꾸지 말 것** — 채점자가 같은 결과를 재현해야 한다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(20260302)   # ⚠️ 이 줄은 바꾸지 마세요

N_TRIALS = 100_000                      # 문제 3에서 쓸 시뮬레이션 횟수

print(f"시뮬레이션 {N_TRIALS:,}회 준비 완료")

## 문제 1 — 모형 세우기 (5점)

### (a) 게임 규칙을 3~5줄로 서술한다

이 셀을 더블클릭해 작성한다. 확률의 **출처 또는 가정**을 반드시 밝힐 것.

> **게임 이름:** (여기에 작성)
>
> **규칙:** (여기에 작성)
>
> **확률의 출처 / 가정:** (여기에 작성)

### (b) 확률변수 $X$ 를 한 문장으로 정의한다

> **$X$ =** (여기에 작성 — 예: "한 판의 순수익, 단위 원")

### ✏️ TODO 1 — PMF 를 배열 두 개로 적는다

⚠️⚠️ **아래는 예시일 뿐이다.** 동전 던지기라 너무 단순해서
**그대로 제출하면 문제 1 점수를 받을 수 없다.** 반드시 자기 게임으로 바꿀 것.

예시를 남겨 둔 이유는 **처음 실행했을 때 전체가 돌아가는 것을 보여주기 위해서**다.
먼저 한 번 끝까지 실행해 보고, 그다음 이 셀만 자기 게임으로 갈아끼우면 된다.

In [ ]:
# TODO 1: 자기 게임의 PMF 로 바꾸세요.
#         values - X 가 가질 수 있는 값들
#         probs  - 각 값의 확률 (합이 정확히 1이어야 한다)
#
# ⚠️ 아래는 "동전 던지기 앞면 +1 / 뒷면 -1" 예시입니다. 반드시 바꾸세요.
values = np.array([1.0, -1.0])
probs = np.array([0.5, 0.5])

assert np.all(probs >= 0), "확률은 음수일 수 없다"
assert abs(probs.sum() - 1.0) < 1e-9, f"확률의 합이 1이 아니다: {probs.sum()}"

print("PMF")
for v, p in zip(values, probs):
    print(f"  P[X = {v:>10,.1f}] = {p:.4f}")
print(f"  합 = {probs.sum():.4f}")

> 💡 값이 많은 게임(예: 강화 시도 횟수)이라면 `np.arange` 로 만들어도 된다.
> 중요한 것은 **`values` 와 `probs` 의 길이가 같고 확률의 합이 1** 이라는 것뿐이다.

## 문제 2 — 손으로 계산 (8점)

### ✏️ TODO 2 — 기댓값과 분산을 정의대로 구한다

$$\mathbb{E}[X] = \sum_x x\,p_X(x) \qquad
\mathbb{E}[X^2] = \sum_x x^2 p_X(x) \qquad
\mathrm{Var}[X] = \mathbb{E}[X^2] - (\mathbb{E}[X])^2$$

⚠️ $\mathbb{E}[X^2]$ 를 $(\mathbb{E}[X])^2$ 로 계산하지 말 것 — **LOTUS 를 쓴다.**

In [ ]:
# TODO 2: 아래 네 값을 정의대로 계산하세요.
#         힌트 - (values * probs).sum() 이 곧 sum(x * p(x)) 입니다
EX = 0.0        # <- E[X]
EX2 = 0.0       # <- E[X^2]   ⚠️ EX ** 2 가 아닙니다
VarX = EX2 - EX ** 2
sigma = VarX ** 0.5

print(f"  E[X]     = {EX:>12,.4f}")
print(f"  E[X^2]   = {EX2:>12,.4f}")
print(f"  Var[X]   = {VarX:>12,.4f}")
print(f"  sigma    = {sigma:>12,.4f}")

### (c) 계산 과정을 손으로 적는다 (숫자만 적으면 감점)

위 코드가 한 일을 **식으로** 적는다. 이 셀을 더블클릭해 작성한다.

> **$\mathbb{E}[X]$ =** (여기에 작성 — 예: $0.1 \times 9000 + 0.9 \times (-1000) = 0$)
>
> **$\mathbb{E}[X^2]$ =** (여기에 작성)
>
> **$\mathrm{Var}[X]$ =** (여기에 작성)

## 문제 3 — 시뮬레이션 검증 (7점)

### ✏️ TODO 3 — 10만 번 뽑는다

`rng.choice` 에 `p=probs` 를 주면 PMF 대로 뽑아 준다.

In [ ]:
# TODO 3: values 에서 probs 대로 N_TRIALS 개를 뽑으세요
#         힌트 - rng.choice(values, size=N_TRIALS, p=probs)
sample = np.zeros(N_TRIALS)      # <- 이 줄을 고치세요

print(f"  표본 크기 : {sample.size:,}")
print(f"  표본평균  : {sample.mean():>12,.4f}   (이론 {EX:,.4f})")
print(f"  표본분산  : {sample.var():>12,.4f}   (이론 {VarX:,.4f})")

### ✏️ TODO 4 — 이론값과 표본값을 표로 비교

> 💡 `np.var(x)` 는 $n$ 으로 나눈다. 표본분산을 $n-1$ 로 나누려면 `np.var(x, ddof=1)`.
> 10만 개에서는 차이가 거의 없다. **이 차이는 Topic 12 에서 다룬다.**

In [ ]:
# TODO 4: 이론값과 표본값을 나란히 담은 rows 를 완성하세요
#         힌트 - ("E[X]", EX, sample.mean()) 처럼 (이름, 이론값, 표본값) 세 쌍
rows = [
    ("E[X]", EX, 0.0),          # <- 표본값을 채우세요
    ("Var[X]", VarX, 0.0),      # <- 표본값을 채우세요
    ("sigma", sigma, 0.0),      # <- 표본값을 채우세요
]

print(f"{'항목':<10}{'이론값':>18}{'표본값':>18}{'차이':>16}")
print("-" * 62)
for name, theo, emp in rows:
    print(f"{name:<10}{theo:>18,.4f}{emp:>18,.4f}{emp - theo:>16,.4f}")

### ✏️ TODO 5 — 누적 평균이 $\mathbb{E}[X]$ 로 수렴하는 그래프

x축은 **로그 스케일**로 두면 초반의 큰 흔들림이 잘 보인다.

> 📌 라벨은 **영어**로. Colab 에는 한글 폰트가 없어 글자가 □□□ 로 깨진다.

In [ ]:
n = np.arange(1, N_TRIALS + 1)

# TODO 5: 처음 n개까지의 누적 평균을 구하세요
#         힌트 - np.cumsum(sample) / n
running = np.zeros(N_TRIALS)     # <- 이 줄을 고치세요

plt.figure(figsize=(7.5, 4))
plt.plot(n, running, lw=1.2, label="Running average")

plt.axhline(EX, color="red", ls="--", lw=1.6, label=f"E[X] = {EX:,.2f}")
plt.xscale("log")
plt.xlabel("Number of trials (log scale)")
plt.ylabel("Running average of X")
plt.title("Does the sample mean settle at E[X]?")
plt.legend()
plt.grid(alpha=0.3, which="both")
plt.show()

print(f"  100회   누적평균 : {running[99]:>12,.4f}")
print(f"  10,000회         : {running[9_999]:>12,.4f}")
print(f"  {N_TRIALS:,}회      : {running[-1]:>12,.4f}")

## 문제 4 — 해석 (5점)

세 물음에 **각각 2~3문장**으로 답한다. 이 셀을 더블클릭해 작성한다.

---

### (a) 이 게임은 참가자에게 유리한가, 불리한가? 근거는?

$\mathbb{E}[X]$ 의 부호를 근거로 말할 것.

> **답:** (여기에 작성)

---

### (b) 몇 판쯤 해야 표본평균을 믿을 수 있겠는가?

$\sigma_X$ 와 위 그래프를 근거로 논한다.
정확한 답은 **Topic 10** 에서 배운다 — 여기서는 그래프를 근거로 한 관찰이면 충분하다.

> **답:** (여기에 작성)

---

### (c) 기댓값만 보고 판단하면 위험한 이유를 이 게임의 예로 설명한다

> **답:** (여기에 작성)

---

## ✅ 제출 전 점검

- [ ] 맨 위에 **학번**과 **고른 게임**을 적었다
- [ ] ⚠️ **PMF 를 예시에서 자기 게임으로 바꿨다** (안 바꾸면 문제 1 점수 없음)
- [ ] `TODO 1~5` 를 모두 채웠다
- [ ] 문제 1(a)(b) · 2(c) · 4(a)(b)(c) 의 **서술을 작성했다**
- [ ] `런타임 → 모두 실행` 으로 **모든 출력이 남아 있다**
- [ ] 그래프에 축 이름 · 제목 · 범례가 있고 **한글이 없다**
- [ ] 파일명을 **`HW_학번_T03.ipynb`** 로 바꿨다

**제출처와 기한은 PLATO · Google Classroom 공지를 확인한다.**

### 자주 하는 실수

- $\mathbb{E}[X^2]$ 를 $(\mathbb{E}[X])^2$ 로 계산 → **LOTUS 를 쓸 것**
- 확률의 합이 1 이 아닌 채로 진행 → 위 `assert` 가 잡아 준다
- 그래프 축 라벨을 한글로 → Colab 에서 깨진다